# ABPT Anchor V1 — Colab notebook (legacy-bootstrap style)

Это вариант **в стиле старого colab notebook**, но под текущий Anchor V1 pipeline.

Почему он полезнее обычного минимального notebook:
- умеет работать через **Google Drive**
- умеет **клонировать repo**
- умеет **загрузить zip** репозитория прямо в Colab
- сразу готовит артефакты для текущего train/eval/report loop

Цель notebook:
1. поднять репозиторий в Colab
2. проверить GPU runtime
3. прогнать короткий `world-like` run на `shakespeare`
4. прогнать короткий `anchor-synthetic` run
5. сохранить history/report артефакты на Drive


In [ ]:
# @title 1. Repo bootstrap (Drive / ZIP / Git clone)
from pathlib import Path
import shutil
import zipfile
import os

USE_DRIVE = False
UPLOAD_ZIP = False
CLONE_REPO = False

DRIVE_REPO_PATH = '/content/drive/MyDrive/ABPT'
COLAB_REPO_PATH = '/content/ABPT'
REPO_GIT_URL = ''  # e.g. 'https://github.com/<user>/<repo>.git'
ZIP_TARGET = '/content/ABPT.zip'

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

repo_path = Path(COLAB_REPO_PATH)

if repo_path.exists():
    print('Repo already present:', repo_path)
elif USE_DRIVE:
    src = Path(DRIVE_REPO_PATH)
    if not src.exists():
        raise FileNotFoundError(f'Drive repo not found: {src}')
    shutil.copytree(src, repo_path)
    print('Copied repo from Drive ->', repo_path)
elif UPLOAD_ZIP:
    from google.colab import files
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError('No zip uploaded')
    uploaded_name = next(iter(uploaded.keys()))
    shutil.move(uploaded_name, ZIP_TARGET)
    with zipfile.ZipFile(ZIP_TARGET, 'r') as zf:
        zf.extractall('/content')
    if not repo_path.exists():
        extracted_dirs = [p for p in Path('/content').iterdir() if p.is_dir() and (p / 'train.py').exists()]
        if extracted_dirs:
            extracted_dirs[0].rename(repo_path)
    print('Extracted zip ->', repo_path)
elif CLONE_REPO:
    if not REPO_GIT_URL:
        raise RuntimeError('Set REPO_GIT_URL before clone')
    !git clone "$REPO_GIT_URL" "$COLAB_REPO_PATH"
else:
    raise RuntimeError('Choose one bootstrap mode: USE_DRIVE / UPLOAD_ZIP / CLONE_REPO')

%cd /content/ABPT
print('Working dir:', Path.cwd())


In [ ]:
# @title 2. Dependencies
import sys
!{sys.executable} -m pip install -q --upgrade pip
!{sys.executable} -m pip install -q torch einops pytest


In [ ]:
# @title 3. Runtime / GPU check
import torch, platform, json, os
info = {
    'python': platform.python_version(),
    'torch': torch.__version__,
    'cuda_available': torch.cuda.is_available(),
    'device_count': torch.cuda.device_count(),
    'devices': [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
    'cwd': os.getcwd(),
}
print(json.dumps(info, indent=2))
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', DEVICE)


In [ ]:
# @title 4. Optional smoke tests
RUN_TESTS = False
if RUN_TESTS:
    !pytest -q tests/test_abpt_anchor_v1.py tests/test_train_eval.py tests/test_anchor_synthetic.py
else:
    print('Smoke tests skipped')


## Phase A — world-like warmup

Это короткий sanity run на `tiny_shakespeare`.

Если хочешь реальный world warmup:
- поставь GPU runtime
- увеличь `WORLD_STEPS` хотя бы до 1000+
- потом уже переходи к synthetic/conflict stage


In [ ]:
# @title 5. World-like Shakespeare run
from pathlib import Path

WORLD_PRESET = 'toy'
WORLD_STEPS = 200
WORLD_HISTORY = Path('archive/colab_world_history.json')
WORLD_REPORT = Path('docs/research/colab_world_report.md')

!python train.py --preset {WORLD_PRESET} --stage anchor --dataset shakespeare --device {DEVICE} --steps {WORLD_STEPS} --history_path {WORLD_HISTORY}
!python scripts/generate_anchor_training_report.py {WORLD_HISTORY} --output {WORLD_REPORT}
print(WORLD_REPORT)


In [ ]:
# @title 6. Display world report
from pathlib import Path
from IPython.display import Markdown, display
p = Path('docs/research/colab_world_report.md')
if p.exists():
    display(Markdown(p.read_text(encoding='utf-8')))
else:
    print('World report missing')


## Phase B/C — anchor-synthetic curriculum

Это текущий theory-aligned synthetic dataset.

Важно: если даже здесь `proposal_influence == 0`, значит следующий bottleneck — не просто данные, а explicit supervision на proposal/revision behavior.


In [ ]:
# @title 7. Synthetic anchor run
from pathlib import Path

SYN_PRESET = 'toy'
SYN_STEPS = 200
SYN_HISTORY = Path('archive/colab_anchor_synth_history.json')
SYN_REPORT = Path('docs/research/colab_anchor_synth_report.md')

!python train.py --preset {SYN_PRESET} --stage anchor --dataset anchor-synthetic --device {DEVICE} --steps {SYN_STEPS} --history_path {SYN_HISTORY}
!python scripts/generate_anchor_training_report.py {SYN_HISTORY} --output {SYN_REPORT}
print(SYN_REPORT)


In [ ]:
# @title 8. Display synthetic report
from pathlib import Path
from IPython.display import Markdown, display
p = Path('docs/research/colab_anchor_synth_report.md')
if p.exists():
    display(Markdown(p.read_text(encoding='utf-8')))
else:
    print('Synthetic report missing')


In [ ]:
# @title 9. Save artifacts back to Drive (optional)
from pathlib import Path
import shutil

SAVE_TO_DRIVE = USE_DRIVE
DRIVE_ARTIFACT_DIR = Path('/content/drive/MyDrive/ABPT_colab_runs')
ARTIFACTS = [
    Path('archive/colab_world_history.json'),
    Path('archive/colab_anchor_synth_history.json'),
    Path('docs/research/colab_world_report.md'),
    Path('docs/research/colab_anchor_synth_report.md'),
]

if SAVE_TO_DRIVE:
    DRIVE_ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
    for artifact in ARTIFACTS:
        if artifact.exists():
            dst = DRIVE_ARTIFACT_DIR / artifact.name
            shutil.copy2(artifact, dst)
            print('Saved:', dst)
else:
    print('Drive export skipped')
